In [ ]:
# We are building a list of synonyms for the medicine, so that eg. panodil and paracetamol are not counted twice as two seperate medicines in one row.
# In stead they are counted as one medicin: paracetamol.
# We build the list from the files:  
# `Afregistrerede_Laegemidler.xls`
# `ListeOverGodkendteLaegemidler.xlsx`

In [ ]:
import pandas as pd
import re, json
from collections import defaultdict

###############################################################################
# Helper: normalise every string in exactly the same way
###############################################################################
_CLEAN_RX = re.compile(r'"[^"]*"')

def clean(text: str) -> str:
    """Remove quoted substrings, squeeze whitespace, lower-case."""
    text = _CLEAN_RX.sub('', text)          # strip “…”
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

###############################################################################
# 1. Read the two Excel sheets
###############################################################################
discontinued = pd.read_excel(
    'Afregistrerede_Laegemidler.xls',
    sheet_name='AfregLægemidlerSidsteÅr',
    usecols=['Lægemiddel','AktiveSubstanser']           # read only what we need
)

current = pd.read_excel(
    'ListeOverGodkendteLaegemidler.xlsx',
    sheet_name='Godkendte Lægemidler',
    usecols=['Navn','AktiveSubstanser']
)

###############################################################################
# 2. Build: generic  ➜  {synonym, synonym, …}
###############################################################################
generic_to_synonyms: dict[str, set[str]] = defaultdict(set)

# discontinued sheet
for syn, gen in zip(discontinued['Lægemiddel'], discontinued['AktiveSubstanser']):
    if pd.notna(syn) and pd.notna(gen):
        generic_to_synonyms[clean(gen)].add(clean(syn))

# current sheet
for syn, gen in zip(current['Navn'], current['AktiveSubstanser']):
    if pd.notna(syn) and pd.notna(gen):
        generic_to_synonyms[clean(gen)].add(clean(syn))

###############################################################################
# 3. (optional) reverse map: synonym ➜ generic
###############################################################################
synonym_to_generic = {
    syn : gen
    for gen, synset in generic_to_synonyms.items()
    for syn in synset
}

###############################################################################
# 4. Write JSON: convert each set to a sorted list for serialisation
###############################################################################
json_ready = {gen: sorted(list(syns)) for gen, syns in generic_to_synonyms.items()}

with open('generic_to_synonyms.json', 'w', encoding='utf-8') as fh:
    json.dump(json_ready, fh, ensure_ascii=False, indent=2)

# If you also want the reverse mapping:
# with open('synonym_to_generic.json', 'w', encoding='utf-8') as fh:
#     json.dump(synonym_to_generic, fh, ensure_ascii=False, indent=2)